# POC – MCP Tool Registration

This example aspires to verify the points listed in [POC - AI with Splunk Apps](https://cisco-my.sharepoint.com/:w:/r/personal/hbalacha_cisco_com/Documents/POC%20-%20AI%20with%20Splunk%20Apps.docx?d=w2776e089011943abbd84c0fa30a53f34&csf=1&web=1&e=RxvShR)

- Develop @tool Decorator
  - [ ] Capture e.g. tool_name, description, inputs, outputs
- MCP JSONSchema can (and most probably should) be used for tool registration in Splunk


In [ ]:
from fastmcp import Client
from mcp.types import Tool

MCP_SERVER_HOST: str = "0.0.0.0"
MCP_SERVER_PORT: int = 2137


conn_str = f"http://{MCP_SERVER_HOST}:{MCP_SERVER_PORT}/mcp"
mcp_client = Client("./tools.py")


async def get_tools() -> list[Tool]:
    tools = []
    async with mcp_client:
        tools = await mcp_client.list_tools()

    return tools


call_tool_result = await get_tools()
for tool in call_tool_result:
    print(tool.name)
    print(tool.description)
    print(tool.inputSchema)
    print(tool.outputSchema)
    print(tool.meta)

- execution_mode (external_http)
  - Is this what MCP calls `transports`?
- execution_metadata (endpoint URL)
  - Isn't that just `/execute_tool` or `tool/call`?

In [ ]:
mcp_client = Client("tools.py")


async def call_tool(tool_name: str, arguments: dict[str, int]):
    result = None
    async with mcp_client:
        result = await mcp_client.call_tool(tool_name, arguments)

    return result


call_tool_result = await call_tool("generating_csc", {"count": 10})

[print(data) for data in call_tool_result.data]


- [ ] Support YAML v2 Tool Definition
- [ ] Merge decorator + YAML for complete metadata

- Implement Post-Install Script (sdk_post_install.py):
  - [ ] Load app modules → decorators populate RegisteredTools
  - [ ] Merge YAML/decorator metadata
  - [ ] Build MCP /tools/register payload
  - [ ] Call MCP registry with credentials (from App Manager)
  - [ ] Log success/fail to MCP audit


In [ ]:
import configparser
import os
from dataclasses import asdict, dataclass, field
from typing import Any, Literal

from mcp.types import Tool


@dataclass
class SplunkMeta:
    permissions: list[str]
    tool_type: str
    schema_version: str


@dataclass
class McpInputOutputSchema:
    type: Literal["object"] = "object"
    properties: dict[str, Any] = field(default_factory=lambda: {})  # pyright: ignore[reportExplicitAny]
    required: list[str] = field(default_factory=lambda: [])


tool_reg_prefix = "app:mcp_tool"


def filter_sections(section_name: str):
    return section_name.startswith(tool_reg_prefix)


def match_input_schema(input: Literal["query_string"] | Literal["other"]):
    match input:
        case "query_string":
            return {
                "type": "object",
                "properties": {
                    "query_string": {
                        "type": "string",
                        "description": "SPL2 query string",
                    }
                },
            }
        case _:
            raise NotImplementedError("We don't know what to put here lol")


def parse_app_conf_tool_registrations(file_path: str) -> list[Tool]:
    config = configparser.ConfigParser()
    all_sections_len = config.read(file_path)
    if len(all_sections_len) == 0:
        return []

    tool_reg_sections: list[str] = list(filter(filter_sections, config.sections()))
    if len(tool_reg_sections) == 0:
        return []

    ini_tools: list[Tool] = []
    for reg_section in tool_reg_sections:
        reg_section_data = config[reg_section]

        name: str = reg_section.split(":")[2]
        description = reg_section_data["description"]
        # https://modelcontextprotocol.io/specification/2025-06-18/schema#tool
        inputSchema = McpInputOutputSchema(properties={}, required=[])
        outputSchema = McpInputOutputSchema(properties={}, required=[])
        meta = SplunkMeta(
            permissions=[
                perm.strip()
                for perm in reg_section_data["permissions"].strip().split(",")
            ],
            tool_type="search",
            schema_version=reg_section_data["schema_version"].strip(),
        )

        ini_tool = Tool(
            name=name,
            description=description,
            inputSchema=asdict(inputSchema),
            outputSchema=asdict(outputSchema),
            _meta=asdict(meta),
        )
        ini_tools.append(ini_tool)

    return ini_tools


async def post_install():
    curr_path = os.path.join(os.getcwd(), "..", "default", "app.conf")
    print(curr_path)
    yaml_tool_registrations: list[Tool] = parse_app_conf_tool_registrations(curr_path)
    [print(toolReg) for toolReg in yaml_tool_registrations]


await post_install()


/Users/bjedreck/Projects/spl-mcp-tool/bin/../default/app.conf
name='aws_logs_search' title=None description='Execute SPL queries against AWS logs' inputSchema={'type': 'object', 'properties': {}, 'required': []} outputSchema={'type': 'object', 'properties': {}, 'required': []} icons=None annotations=None meta={'permissions': ['role:search_admin', 'role:aws_analyst'], 'tool_type': 'search', 'schema_version': '1.0'}
